# 02 - Mining per segment

Phase 2. Mines the cleaned BPI 2019 log per `Item Category` segment and writes
the evidence store the Phase 3 agent will reason over. Every number below is
computed by pandas/pm4py; the summary sentences are generated by code.

Run artefacts land in `outputs/runs/<run_id>/<segment-slug>/` (`as_is.bpmn`,
`dfg.json`, `dfg_view.json`, `evidence.json`), plus a run-level
`evidence.json` and `manifest.json` (settings + library versions).

In [1]:
import time

import pandas as pd

from processpilot.config import find_project_root, load_env, resolve_source_path
from processpilot.ingest.mapping import apply_mapping, load_preset, split_by_segment
from processpilot.ingest.validate import apply_cleaning, validate
from processpilot.mining.evidence import format_duration
from processpilot.mining.run import mine_segment, new_run_id, write_run
from processpilot.mining.settings import MiningSettings
from processpilot.sources.file_source import FileSource

load_env()
mapping, preset = load_preset("bpi2019")
settings = MiningSettings.from_preset(preset)
pd.set_option("display.max_colwidth", 80)

## Ingest (same steps as notebook 01)

Load through the parquet cache, map, validate, and apply the preset's
suggested cleaning policy. Mining always runs on the cleaned log.

In [2]:
source = FileSource(resolve_source_path(preset))
canonical = apply_mapping(source.fetch(), mapping)
v = preset["validation"]
bounds = {
    "plausible_timestamp_min": v["plausible_timestamp_min"],
    "plausible_timestamp_max": v["plausible_timestamp_max"],
}
report = validate(canonical, **bounds)
clean, clean_report = apply_cleaning(
    canonical, report, **preset["suggested_cleaning_policy"], **bounds
)
segments = split_by_segment(clean)
print(f"{clean_report.event_count:,} events, {clean_report.case_count:,} cases, "
      f"{len(segments)} segments")

1,593,429 events, 251,654 cases, 4 segments


## Mine every segment and write the run

`mine_segment` is pure; `write_run` is the only step that touches disk.

In [3]:
results = []
for name, seg_df in segments.items():
    start = time.perf_counter()
    results.append(mine_segment(seg_df, name, settings))
    print(f"{name:34s} {time.perf_counter() - start:6.1f}s  "
          f"{len(results[-1].evidence.items)} evidence items")

run_dir = find_project_root() / "outputs" / "runs" / new_run_id()
store = write_run(results, run_dir, settings, source.describe())
print(f"\n{len(store.items)} evidence items written to {run_dir}")

2-way match                           9.8s  54 evidence items


3-way match, invoice after GR        25.9s  61 evidence items


3-way match, invoice before GR      127.8s  60 evidence items


Consignment                           4.1s  55 evidence items



230 evidence items written to C:\Users\user\Desktop\Thesis\Process Pilot project\outputs\runs\20261016T110043Z


## Top bottlenecks per segment

Ranked by impact = median wait x share of cases traversing the edge. "Wait"
on `a -> b` is the time from completing `a` to completing `b` (the log has a
single timestamp per event, so it includes `b`'s processing time).

In [4]:
def days(seconds):
    return round(seconds / 86400, 1)

rows = []
for item in store.query(type="bottleneck"):
    m = item.metrics
    if m["rank"] > 5:
        continue
    rows.append({
        "segment": item.segment,
        "id": item.id,
        "transition": f"{m['source']} -> {m['target']}",
        "case share %": round(m["case_share"] * 100, 1),
        "median wait (d)": days(m["median_wait_seconds"]),
        "P90 wait (d)": days(m["p90_wait_seconds"]),
        "impact (d)": days(m["impact_score"]),
    })
pd.DataFrame(rows).set_index(["segment", "id"])

transition  \
segment                        id                                                                                                  
2-way match                    BN-01-2-way-match                    Change Approval for Purchase Order -> Vendor creates invoice   
                               BN-02-2-way-match                            Vendor creates invoice -> Create Purchase Order Item   
                               BN-03-2-way-match                             Clear Invoice -> Change Approval for Purchase Order   
                               BN-04-2-way-match                    Record Invoice Receipt -> Change Approval for Purchase Order   
                               BN-05-2-way-match                                         Record Invoice Receipt -> Clear Invoice   
3-way match, invoice after GR  BN-01-3-way-match-invoice-after-gr                        Record Invoice Receipt -> Clear Invoice   
                               BN-02-3-way-match-invoice-after-gr                 Record Goods Receipt -> Record Invoice Receipt   
                               BN-03-3-way-match-invoice-after-gr               Vendor creates invoice -> Record Invoice Receipt   
                               BN-04-3-way-match-invoice-after-gr           Create Purchase Order Item -> Vendor creates invoice   
                               BN-05-3-way-match-invoice-after-gr                          Remove Payment Block -> Clear Invoice   
3-way match, invoice before GR BN-01-3-way-match-invoice-before-gr                       Record Invoice Receipt -> Clear Invoice   
                               BN-02-3-way-match-invoice-before-gr                Record Goods Receipt -> Record Invoice Receipt   
                               BN-03-3-way-match-invoice-before-gr                         Remove Payment Block -> Clear Invoice   
                               BN-04-3-way-match-invoice-before-gr            Create Purchase Order Item -> Record Goods Receipt   
                               BN-05-3-way-match-invoice-before-gr          Create Purchase Order Item -> Vendor creates invoice   
Consignment                    BN-01-consignment                              Create Purchase Order Item -> Record Goods Receipt   
                               BN-02-consignment                              Receive Order Confirmation -> Record Goods Receipt   
                               BN-03-consignment                                   Create Purchase Order Item -> Change Quantity   
                               BN-04-consignment                                         Change Quantity -> Record Goods Receipt   
                               BN-05-consignment                                    Record Goods Receipt -> Record Goods Receipt   

                                                                    case share %  \
segment                        id                                                  
2-way match                    BN-01-2-way-match                            19.9   
                               BN-02-2-way-match                            40.8   
                               BN-03-2-way-match                            12.1   
                               BN-04-2-way-match                             7.6   
                               BN-05-2-way-match                            25.9   
3-way match, invoice after GR  BN-01-3-way-match-invoice-after-gr           51.7   
                               BN-02-3-way-match-invoice-after-gr           35.5   
                               BN-03-3-way-match-invoice-after-gr           32.7   
                               BN-04-3-way-match-invoice-after-gr           29.3   
                               BN-05-3-way-match-invoice-after-gr           12.6   
3-way match, invoice before GR BN-01-3-way-match-invoice-before-gr          56.4   
                               BN-02-3-way-match-invoice-before-gr          48.0   
                               BN-03-

### The same findings as the agent will read them

These code-generated sentences, with their IDs, are all the Phase 3 agent
sees -- never the event log.

In [5]:
for seg in store.segments():
    print(f"\n== {seg}")
    for item in store.query(segment=seg, type="bottleneck")[:3]:
        print(f"[{item.id}] {item.summary}")


== 2-way match
[BN-01-2-way-match] Transition 'Change Approval for Purchase Order' -> 'Vendor creates invoice' occurs in 19.9% of cases (208) with a median wait of 29.3 days (mean 49.2 days, P90 100.0 days); impact rank 1.
[BN-02-2-way-match] Transition 'Vendor creates invoice' -> 'Create Purchase Order Item' occurs in 40.8% of cases (426) with a median wait of 11.5 days (mean 21.9 days, P90 33.6 days); impact rank 2.
[BN-03-2-way-match] Transition 'Clear Invoice' -> 'Change Approval for Purchase Order' occurs in 12.1% of cases (126) with a median wait of 35.4 days (mean 40.2 days, P90 76.9 days); impact rank 3.

== 3-way match, invoice after GR
[BN-01-3-way-match-invoice-after-gr] Transition 'Record Invoice Receipt' -> 'Clear Invoice' occurs in 51.7% of cases (7,834) with a median wait of 21.1 days (mean 26.9 days, P90 64.0 days); impact rank 1.
[BN-02-3-way-match-invoice-after-gr] Transition 'Record Goods Receipt' -> 'Record Invoice Receipt' occurs in 35.5% of cases (5,380) with a m

## Segment KPIs (baseline for judging a bottleneck)

In [6]:
pd.DataFrame([
    {
        "segment": i.segment,
        "cases": i.metrics["case_count"],
        "events": i.metrics["event_count"],
        "batch repeats collapsed": i.metrics["collapsed_event_count"],
        "median throughput": format_duration(i.metrics["median_throughput_seconds"]),
        "P90 throughput": format_duration(i.metrics["p90_throughput_seconds"]),
    }
    for i in store.query(type="kpi")
]).set_index("segment")

,cases,events,batch repeats collapsed,median throughput,P90 throughput
segment,,,,,
2-way match,1044,5894,4,23.7 days,196.9 days
"3-way match, invoice after GR",15164,139354,177856,63.4 days,133.2 days
"3-way match, invoice before GR",220948,1231234,3003,66.3 days,127.0 days
Consignment,14498,36060,24,19.9 days,48.7 days


## Conformance

Each violation is reported over all evaluated cases and over completed cases
only (last activity in the preset's `case_completion` list). The gap between
the two is mostly cases still running when the log was extracted.

In [7]:
pd.DataFrame([
    {
        "segment": i.segment,
        "rule": f"{i.metrics['rule_id']} {i.metrics['rule_name']}",
        "violations": i.metrics["violation_count"],
        "of evaluated": i.metrics["evaluated_cases"],
        "share %": round(i.metrics["violation_share"] * 100, 1),
        "completed violations": i.metrics["completed_violation_count"],
        "of completed": i.metrics["completed_evaluated_cases"],
        "completed share %": None if i.metrics["completed_violation_share"] is None
        else round(i.metrics["completed_violation_share"] * 100, 1),
    }
    for i in store.query(type="conformance")
]).set_index(["segment", "rule"])

violations  \
segment                        rule                                                    
2-way match                    R-02 invoice_recorded_before_cleared                0   
                               R-03 invoice_must_occur                           357   
3-way match, invoice after GR  R-01 three_way_match_order                          0   
                               R-02 invoice_recorded_before_cleared                7   
                               R-03 invoice_must_occur                          4054   
                               R-04 goods_receipt_required_for_three_way         643   
3-way match, invoice before GR R-02 invoice_recorded_before_cleared               17   
                               R-03 invoice_must_occur                         21446   
                               R-04 goods_receipt_required_for_three_way       14536   

                                                                          of evaluated  \
segment                        rule                                                      
2-way match                    R-02 invoice_recorded_before_cleared                303   
                               R-03 invoice_must_occur                            1044   
3-way match, invoice after GR  R-01 three_way_match_order                        11110   
                               R-02 invoice_recorded_before_cleared               9656   
                               R-03 invoice_must_occur                           15164   
                               R-04 goods_receipt_required_for_three_way         15164   
3-way match, invoice before GR R-02 invoice_recorded_before_cleared             173254   
                               R-03 invoice_must_occur                          220948   
                               R-04 goods_receipt_required_for_three_way        220948   

                                                                          share %  \
segment                        rule                                                 
2-way match                    R-02 invoice_recorded_before_cleared           0.0   
                               R-03 invoice_must_occur                       34.2   
3-way match, invoice after GR  R-01 three_way_match_order                     0.0   
                               R-02 invoice_recorded_before_cleared           0.1   
                               R-03 invoice_must_occur                       26.7   
                               R-04 goods_receipt_required_for_three_way      4.2   
3-way match, invoice before GR R-02 invoice_recorded_before_cleared           0.0   
                               R-03 invoice_must_occur                        9.7   
                               R-04 goods_receipt_required_for_three_way      6.6   

                                                                          completed violations  \
segment                        rule                                                              
2-way match                    R-02 invoice_recorded_before_cleared                          0   
                               R-03 invoice_must_occur                                       1   
3-way match, invoice after GR  R-01 three_way_match_order                                    0   
                               R-02 invoice_recorded_before_cleared                          4   
                               R-03 invoice_must_occur                                     303   
                               R-04 goods_receipt_required_for_three_way                   292   
3-way match, invoice before GR R-02 invoice_recorded_before_cleared                          6   
                               R-03 invoice_must_occur                                    7760   
                               R-04 goods_receipt_required_for_three_way                  7866   

                                                                          of completed  \
segment  

## Rework and handovers (top 3 per segment)

In [8]:
for seg in store.segments():
    print(f"\n== {seg}")
    for item in store.query(segment=seg, type="rework")[:3]:
        print(f"[{item.id}] {item.summary}")
    handovers = [i for i in store.query(segment=seg, type="resource") if i.id.startswith("HO-")]
    for item in handovers[:3]:
        print(f"[{item.id}] {item.summary}")


== 2-way match
[RW-01-2-way-match] 'Change Approval for Purchase Order' is repeated within 42.9% of cases (448; 2,063 extra executions); those cases take a median 52.8 days longer than cases performing it once (association, not proven cause).
[RW-02-2-way-match] 'Record Invoice Receipt' is repeated within 0.8% of cases (8; 8 extra executions); those cases take a median 17.0 days longer than cases performing it once (association, not proven cause).
[RW-03-2-way-match] 'Vendor creates invoice' is repeated within 0.2% of cases (2; 2 extra executions); those cases take a median 3.2 days longer than cases performing it once (association, not proven cause).
[HO-00-2-way-match] Across all transitions, the median wait is 1.7 days when the resource changes and 0 s when it does not (0 events have no resource).
[HO-01-2-way-match] On 'Change Approval for Purchase Order' -> 'Vendor creates invoice', 100.0% of transitions hand over to a different resource (19.9% of cases); median wait 29.3 days wi

## Run artefacts

In [9]:
sorted(str(p.relative_to(run_dir)) for p in run_dir.rglob("*") if p.is_file())

['2-way-match\\as_is.bpmn',
 '2-way-match\\dfg.json',
 '2-way-match\\dfg_view.json',
 '2-way-match\\evidence.json',
 '3-way-match-invoice-after-gr\\as_is.bpmn',
 '3-way-match-invoice-after-gr\\dfg.json',
 '3-way-match-invoice-after-gr\\dfg_view.json',
 '3-way-match-invoice-after-gr\\evidence.json',
 '3-way-match-invoice-before-gr\\as_is.bpmn',
 '3-way-match-invoice-before-gr\\dfg.json',
 '3-way-match-invoice-before-gr\\dfg_view.json',
 '3-way-match-invoice-before-gr\\evidence.json',
 'consignment\\as_is.bpmn',
 'consignment\\dfg.json',
 'consignment\\dfg_view.json',
 'consignment\\evidence.json',
 'evidence.json',
 'manifest.json']